In [1]:
print("Phishing URL Detection Project")
print("Google Colab is working successfully!")

Phishing URL Detection Project
Google Colab is working successfully!


In [2]:

import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
except ImportError:
    print("TensorFlow is not installed.")

print("Environment check completed.")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
TensorFlow version: 2.21.0
Environment check completed.


In [3]:
import pandas as pd
import requests
from pathlib import Path

dataset_url = (
    "https://zenodo.org/records/19371661/files/"
    "balanced_urls.csv?download=1"
)

dataset_path = Path("/content/balanced_urls.csv")

print("Downloading dataset...")

response = requests.get(dataset_url, timeout=120)
response.raise_for_status()

dataset_path.write_bytes(response.content)

print("Download completed!")
print("File size:", round(dataset_path.stat().st_size / (1024 * 1024), 2), "MB")

Download completed!
File size: 29.35 MB


In [4]:
import pandas as pd

df = pd.read_csv("/content/balanced_urls.csv")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))

Dataset shape: (340000, 2)

Column names:
['label', 'url']

First 5 rows:


,label,url
0,phishing,http://guiadebelezadejpa.com/Paypal_Vireficati...
1,legitimate,https://www.actionagainsthunger.org/story/empo...
2,legitimate,http://host.sk/faq/
3,legitimate,https://www.tebyan.net/index.aspx?pid=18389
4,legitimate,http://www.kochava.com/fraud-for-networks/



Missing values:
label    0
url      0
dtype: int64

Label distribution:
label
phishing      170000
legitimate    170000
Name: count, dtype: int64


In [5]:
import pandas as pd

# Load the dataset
df = pd.read_csv("/content/balanced_urls.csv")

# 1. Check duplicate URLs
print("Duplicate URLs:", df["url"].duplicated().sum())

# 2. Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# 3. Check unique labels
print("\nUnique labels:")
print(df["label"].value_counts(dropna=False))

# 4. Check empty URLs
print("\nEmpty URLs:", df["url"].isna().sum())

# 5. Check URL lengths
url_lengths = df["url"].astype(str).str.len()

print("\nURL Length Statistics:")
print(url_lengths.describe())

# 6. Check whether URLs contain leading/trailing spaces
spaces = (
    df["url"].astype(str)
    != df["url"].astype(str).str.strip()
)

print("\nURLs with extra spaces:", spaces.sum())

print("\nInitial data quality check completed!")

Duplicate URLs: 277
Duplicate rows: 277

Unique labels:
label
phishing      170000
legitimate    170000
Name: count, dtype: int64

Empty URLs: 0

URL Length Statistics:
count    340000.000000
mean         79.511609
std          56.541229
min           4.000000
25%          47.000000
50%          66.000000
75%          93.000000
max        2337.000000
Name: url, dtype: float64

URLs with extra spaces: 0

Initial data quality check completed!


In [6]:
import pandas as pd

df = pd.read_csv("/content/balanced_urls.csv")

# Inspect duplicate URLs
duplicate_urls = df[df["url"].duplicated(keep=False)]

print("Rows containing duplicate URLs:", len(duplicate_urls))

# Check whether any URL has conflicting labels
label_counts = df.groupby("url")["label"].nunique()

conflicting_urls = label_counts[label_counts > 1]

print("URLs with conflicting labels:", len(conflicting_urls))

# Display a few duplicate examples
print("\nSample duplicate records:")
display(duplicate_urls.sort_values("url").head(10))

Rows containing duplicate URLs: 422
URLs with conflicting labels: 0

Sample duplicate records:


,label,url
333299,phishing,&.rand=13vqcr8bp0gud&lc=1033&id=64855&mkt=en-u...
191717,phishing,&.rand=13vqcr8bp0gud&lc=1033&id=64855&mkt=en-u...
314876,phishing,http://184.82.146.86/hotmail/login.srfwawsigni...
21910,phishing,http://184.82.146.86/hotmail/login.srfwawsigni...
140972,phishing,http://200.35.90.92:84/ws/eBayISAPI%3bdllSignI...
42990,phishing,http://200.35.90.92:84/ws/eBayISAPI%3bdllSignI...
197024,phishing,http://202.82.239.254:82/signin.ebay.com/1.html
39053,phishing,http://202.82.239.254:82/signin.ebay.com/1.html
24439,phishing,http://204.93.167.154/~safetly4/paypal/cycgi-b...
227602,phishing,http://204.93.167.154/~safetly4/paypal/cycgi-b...


In [7]:
import pandas as pd

df = pd.read_csv("/content/balanced_urls.csv")

# Count how many times each URL appears
url_counts = df["url"].value_counts()

# URLs appearing more than once
duplicate_groups = url_counts[url_counts > 1]

print("Total dataset rows:", len(df))
print("Unique URLs:", df["url"].nunique())
print("Duplicate URL groups:", len(duplicate_groups))
print("Rows belonging to duplicate groups:",
      int(duplicate_groups.sum()))
print("Extra duplicate rows:",
      int(duplicate_groups.sum() - len(duplicate_groups)))

# Inspect the most repeated URLs
print("\nMost repeated URLs:")
display(duplicate_groups.head(10))

# Check class balance after keeping one copy of each URL
clean_df = df.drop_duplicates(subset=["url"], keep="first")

print("\nRows after removing duplicate URLs:", len(clean_df))
print("\nLabels after deduplication:")
print(clean_df["label"].value_counts())

Total dataset rows: 340000
Unique URLs: 339723
Duplicate URL groups: 145
Rows belonging to duplicate groups: 422
Extra duplicate rows: 277

Most repeated URLs:


,count
url,
http://alvaroestrella.com/secure/webapps/mpp/home/?cmd=_home&dispa,35
http://caramiaspa.com/css/x554x565/login.html?cmd=_login-run&dispa,18
http://caramiaspa.com/css/x521x415/login.html?cmd=_login-run&dispa,15
http://caramiaspa.com/css/x241x235/login.html?cmd=_login-run&dispa,15
http://www.ghisep.org/fuhhg/Validation/login.php?cmd=login_submit&,7
http://floridadryout.com/15unUUkw/login.html?cmd=_login-run&dispat,6
http://208.115.231.230/pp/webscr.php?cmd=_login-run&dispatch=5885d,6
http://avers-66.ru/Pay/webscr.php?cmd=_login-run&dispatch=5885d80a,6
http://peterspence.com.au/x388x238/account.php?cmd=_login-run&disp,5



Rows after removing duplicate URLs: 339723

Labels after deduplication:
label
legitimate    170000
phishing      169723
Name: count, dtype: int64


In [8]:
import pandas as pd

# Load original dataset
df = pd.read_csv("/content/balanced_urls.csv")

# Remove exact duplicate URLs, keeping one copy
clean_df = df.drop_duplicates(subset=["url"], keep="first").copy()

# Convert labels to numeric values
label_mapping = {
    "legitimate": 0,
    "phishing": 1
}

clean_df["label"] = clean_df["label"].map(label_mapping)

# Verify data
print("Clean dataset shape:", clean_df.shape)
print("\nLabel distribution:")
print(clean_df["label"].value_counts().sort_index())

print("\nAny missing values:")
print(clean_df.isnull().sum())

print("\nRemaining duplicate URLs:",
      clean_df["url"].duplicated().sum())

# Save cleaned dataset
clean_df.to_csv("/content/cleaned_urls.csv", index=False)

print("\nCleaned dataset saved successfully!")

Clean dataset shape: (339723, 2)

Label distribution:
label
0    170000
1    169723
Name: count, dtype: int64

Any missing values:
label    0
url      0
dtype: int64

Remaining duplicate URLs: 0

Cleaned dataset saved successfully!


In [9]:

import pandas as pd
from sklearn.model_selection import train_test_split

# Load the cleaned dataset
df = pd.read_csv("/content/cleaned_urls.csv")

# Separate URLs and labels
X = df["url"].astype(str)
y = df["label"].astype(int)

# First: reserve 10% for final testing
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

# Second: reserve 10% of the full dataset for validation
# 0.111111 of the remaining 90% is approximately 10% overall
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.111111,
    random_state=42,
    stratify=y_trainval
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation label distribution:")
print(y_val.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

# Verify no exact URL overlaps between splits
train_urls = set(X_train)
val_urls = set(X_val)
test_urls = set(X_test)

print("\nTrain-validation overlap:", len(train_urls & val_urls))
print("Train-test overlap:", len(train_urls & test_urls))
print("Validation-test overlap:", len(val_urls & test_urls))


Training samples: 271777
Validation samples: 33973
Testing samples: 33973

Training label distribution:
label
0    136000
1    135777
Name: count, dtype: int64

Validation label distribution:
label
0    17000
1    16973
Name: count, dtype: int64

Testing label distribution:
label
0    17000
1    16973
Name: count, dtype: int64

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [10]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU available:", gpus)
else:
    print("GPU not available. CPU will be used.")


TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
import pandas as pd
import tensorflow as tf

# Load the cleaned dataset
df = pd.read_csv("/content/cleaned_urls.csv")

# Separate training URLs and labels
X_train = df["url"].astype(str)

# Use only the training portion for preprocessing adaptation
from sklearn.model_selection import train_test_split

X_train, X_temp = train_test_split(
    X_train,
    test_size=0.20,
    random_state=42
)

# Create character-level vectorization
max_length = 300

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=100,
    standardize=None,
    split="character",
    output_mode="int",
    output_sequence_length=max_length
)

# Learn the character vocabulary from training URLs only
vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(X_train).batch(1024)
)

# Test the transformation
sample_urls = tf.constant([
    "https://www.wikipedia.org",
    "http://example.com/login"
])

encoded_urls = vectorizer(sample_urls)

print("Character vocabulary:")
print(vectorizer.get_vocabulary())

print("\nEncoded output shape:")
print(encoded_urls.shape)

print("\nFirst URL encoded:")
print(encoded_urls[0].numpy()[:40])

print("\nPreprocessing test completed!")

Character vocabulary:
['', '[UNK]', np.str_('/'), np.str_('t'), np.str_('e'), np.str_('a'), np.str_('o'), np.str_('s'), np.str_('i'), np.str_('c'), np.str_('n'), np.str_('.'), np.str_('p'), np.str_('r'), np.str_('h'), np.str_('m'), np.str_('w'), np.str_('d'), np.str_('l'), np.str_('-'), np.str_('b'), np.str_('u'), np.str_('g'), np.str_('f'), np.str_('1'), np.str_('0'), np.str_('2'), np.str_(':'), np.str_('3'), np.str_('8'), np.str_('5'), np.str_('4'), np.str_('6'), np.str_('9'), np.str_('7'), np.str_('y'), np.str_('k'), np.str_('v'), np.str_('='), np.str_('_'), np.str_('%'), np.str_('x'), np.str_('&'), np.str_('j'), np.str_('z'), np.str_('?'), np.str_('q'), np.str_('I'), np.str_('S'), np.str_('A'), np.str_('P'), np.str_('D'), np.str_('L'), np.str_('B'), np.str_('C'), np.str_('O'), np.str_('F'), np.str_('U'), np.str_('T'), np.str_('E'), np.str_('M'), np.str_('N'), np.str_('G'), np.str_('R'), np.str_('W'), np.str_('V'), np.str_('H'), np.str_('Y'), np.str_('Q'), np.str_('J'), np.str_('Z')

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load cleaned dataset
df = pd.read_csv("/content/cleaned_urls.csv")

X = df["url"].astype(str)
y = df["label"].astype("int32")

# 10% final test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

# 10% of the full dataset for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=1/9,
    random_state=42,
    stratify=y_trainval
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

assert len(set(X_train) & set(X_val)) == 0
assert len(set(X_train) & set(X_test)) == 0
assert len(set(X_val) & set(X_test)) == 0

print("All exact-URL overlap checks passed!")

Training: 271777
Validation: 33973
Testing: 33973
All exact-URL overlap checks passed!


In [13]:
import tensorflow as tf

MAX_LENGTH = 300

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=100,
    standardize=None,
    split="character",
    output_mode="int",
    output_sequence_length=MAX_LENGTH
)

# Adapt only on the actual training split
train_text_ds = tf.data.Dataset.from_tensor_slices(
    X_train.to_numpy()
).batch(1024)

vectorizer.adapt(train_text_ds)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Input sequence length:", MAX_LENGTH)

sample = vectorizer(
    tf.constant(["https://www.wikipedia.org"])
)

print("Encoded shape:", sample.shape)
print("Vectorizer preparation completed!")

Vocabulary size: 97
Input sequence length: 300
Encoded shape: (1, 300)
Vectorizer preparation completed!


In [14]:
import tensorflow as tf

# Model configuration
VOCAB_SIZE = len(vectorizer.get_vocabulary())
EMBEDDING_DIM = 32
MAX_LENGTH = 300

# Build the CNN model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int64),

    tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),

    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    tf.keras.layers.GlobalMaxPooling1D(),

    tf.keras.layers.Dense(64, activation="relu"),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Configure training
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

# Display model architecture
model.summary()

# Test the model's output shape
sample_encoded = vectorizer(
    tf.constant(["https://www.wikipedia.org"])
)

sample_prediction = model(sample_encoded, training=False)

print("\nSample prediction output shape:", sample_prediction.shape)
print("Sample model output:", sample_prediction.numpy())
print("\nCNN model architecture created successfully!")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,633 (68.88 KB)

 Trainable params: 17,633 (68.88 KB)

 Non-trainable params: 0 (0.00 B)


Sample prediction output shape: (1, 1)
Sample model output: [[0.500196]]

CNN model architecture created successfully!


In [15]:
import tensorflow as tf

VOCAB_SIZE = len(vectorizer.get_vocabulary())
MAX_LENGTH = 300
EMBEDDING_DIM = 32

model = tf.keras.Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,), dtype=tf.int64),

    # mask_zero=False avoids passing an unsupported mask to Conv1D
    tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=False
    ),

    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu",
        padding="same"
    ),

    tf.keras.layers.GlobalMaxPooling1D(),

    tf.keras.layers.Dense(64, activation="relu"),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

sample_encoded = vectorizer(
    tf.constant(["https://www.wikipedia.org"])
)

sample_prediction = model(sample_encoded, training=False)

print("\nSample output shape:", sample_prediction.shape)
print("Initial prediction:", sample_prediction.numpy())
print("\nCorrected CNN model created.")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 300, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 300, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,633 (68.88 KB)

 Trainable params: 17,633 (68.88 KB)

 Non-trainable params: 0 (0.00 B)


Sample output shape: (1, 1)
Initial prediction: [[0.4904499]]

Corrected CNN model created.


In [16]:
import tensorflow as tf

BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(urls, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (urls.to_numpy(), labels.to_numpy())
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=20000,
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.map(
        lambda url, label: (
            vectorizer(url),
            tf.cast(label, tf.float32)
        ),
        num_parallel_calls=AUTOTUNE
    )

    return dataset.prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

# Inspect one batch
for batch_urls, batch_labels in train_ds.take(1):
    print("URL batch shape:", batch_urls.shape)
    print("Label batch shape:", batch_labels.shape)
    print("Example labels:", batch_labels[:10].numpy())

print("\nTraining, validation and testing pipelines are ready.")

URL batch shape: (256, 300)
Label batch shape: (256,)
Example labels: [1. 0. 1. 1. 1. 0. 0. 1. 0. 1.]

Training, validation and testing pipelines are ready.


In [17]:

import os
import tensorflow as tf

# Create a directory for saved model artifacts
os.makedirs("/content/phishing_project", exist_ok=True)

# Save the best model weights according to validation loss
checkpoint_path = (
    "/content/phishing_project/best_model.weights.h5"
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=1
)

# Stop if validation loss stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    mode="min",
    verbose=1
)

# Record training metrics for later analysis
csv_logger = tf.keras.callbacks.CSVLogger(
    "/content/phishing_project/training_history.csv"
)

# Train the CNN
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        checkpoint,
        early_stopping,
        csv_logger
    ],
    verbose=1
)

print("\nTraining finished.")
print("Best weights path:", checkpoint_path)
print("Epochs completed:", len(history.history["loss"]))
print("Best validation loss:", min(history.history["val_loss"]))


Epoch 1/10
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8829 - loss: 0.2666 - precision: 0.8657 - recall: 0.9083
Epoch 1: val_loss improved from None to 0.10585, saving model to /content/phishing_project/best_model.weights.h5

Epoch 1: finished saving model to /content/phishing_project/best_model.weights.h5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 23s 18ms/step - accuracy: 0.9284 - loss: 0.1716 - precision: 0.9191 - recall: 0.9393 - val_accuracy: 0.9564 - val_loss: 0.1059 - val_precision: 0.9607 - val_recall: 0.9516
Epoch 2/10
1059/1062 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9590 - loss: 0.1007 - precision: 0.9578 - recall: 0.9602
Epoch 2: val_loss improved from 0.10585 to 0.09037, saving model to /content/phishing_project/best_model.weights.h5

Epoch 2: finished saving model to /content/phishing_project/best_model.weights.h5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - accuracy: 0.9611 - loss: 0.0960 - precision: 0.9600 - recall: 0.9622 - val_accuracy: 0.9629 - val_loss:

In [18]:

import os
import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

checkpoint_path = (
    "/content/phishing_project/best_model.weights.h5"
)

# Confirm that the checkpoint exists
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        "Saved weights not found. Do not retrain yet; "
        "send me this error."
    )

print("Saved weights found!")
print("File size (MB):",
      round(os.path.getsize(checkpoint_path) / (1024 * 1024), 2))

# Load the best saved weights (from the best validation epoch)
model.load_weights(checkpoint_path)
print("\nBest saved weights loaded successfully.")

# Evaluate on the untouched test set
test_results = model.evaluate(test_ds, return_dict=True, verbose=1)

print("\nFINAL TEST METRICS")
for name, value in test_results.items():
    print(f"{name}: {value:.4f}")

# Generate predictions for the test set
y_prob = model.predict(test_ds, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype("int32")

# Recover the true labels in test_ds order
y_true = []
for _, batch_labels in test_ds:
    y_true.extend(batch_labels.numpy().astype("int32"))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["legitimate", "phishing"],
    digits=4,
    zero_division=0
))

print("CONFUSION MATRIX")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


Saved weights found!
File size (MB): 0.23

Best saved weights loaded successfully.
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9694 - loss: 0.0756 - precision: 0.9724 - recall: 0.9662

FINAL TEST METRICS
accuracy: 0.9694
loss: 0.0756
precision: 0.9724
recall: 0.9662
133/133 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  legitimate     0.9665    0.9726    0.9696     17000
    phishing     0.9724    0.9662    0.9693     16973

    accuracy                         0.9694     33973
   macro avg     0.9695    0.9694    0.9694     33973
weighted avg     0.9695    0.9694    0.9694     33973

CONFUSION MATRIX
[[16535   465]
 [  573 16400]]


In [19]:

import os
import json
import tensorflow as tf

PROJECT_DIR = "/content/phishing_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Save model architecture
with open(
    os.path.join(PROJECT_DIR, "model_config.json"),
    "w"
) as f:
    f.write(model.to_json())

# Save the best trained weights
model.save_weights(
    os.path.join(PROJECT_DIR, "best_model.weights.h5")
)

# Save the character vocabulary
vocabulary = [
    str(token) for token in vectorizer.get_vocabulary()
]

with open(
    os.path.join(PROJECT_DIR, "vocabulary.json"),
    "w"
) as f:
    json.dump(vocabulary, f, ensure_ascii=True, indent=2)

# Save essential preprocessing configuration
preprocessing_config = {
    "max_length": 300,
    "max_tokens": 100,
    "split": "character",
    "standardize": None,
    "label_mapping": {
        "legitimate": 0,
        "phishing": 1
    },
    "prediction_threshold": 0.5
}

with open(
    os.path.join(PROJECT_DIR, "preprocessing_config.json"),
    "w"
) as f:
    json.dump(preprocessing_config, f, indent=2)

print("Saved project artifacts:")
for filename in sorted(os.listdir(PROJECT_DIR)):
    path = os.path.join(PROJECT_DIR, filename)
    print(filename, "-", round(os.path.getsize(path) / 1024, 2), "KB")

print("\nArtifact saving completed.")


Saved project artifacts:
best_model.weights.h5 - 236.73 KB
model_config.json - 5.02 KB
preprocessing_config.json - 0.18 KB
training_history.csv - 1.0 KB
vocabulary.json - 0.67 KB

Artifact saving completed.


In [20]:

import os
import json
import numpy as np
import tensorflow as tf

PROJECT_DIR = "/content/phishing_project"

# Reconstruct the CNN architecture
with open(
    os.path.join(PROJECT_DIR, "model_config.json"),
    "r"
) as f:
    model_json = f.read()

reloaded_model = tf.keras.models.model_from_json(model_json)

# Build the model before loading weights
reloaded_model.build((None, 300))

# Load saved weights
reloaded_model.load_weights(
    os.path.join(PROJECT_DIR, "best_model.weights.h5")
)

# Reconstruct the character vectorizer
with open(
    os.path.join(PROJECT_DIR, "vocabulary.json"),
    "r"
) as f:
    saved_vocabulary = json.load(f)

reloaded_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=100,
    standardize=None,
    split="character",
    output_mode="int",
    output_sequence_length=300,
    vocabulary=saved_vocabulary
)

# Run a test prediction
sample_url = "https://www.wikipedia.org"

encoded = reloaded_vectorizer(
    tf.constant([sample_url])
)

probability = float(
    reloaded_model(encoded, training=False).numpy()[0][0]
)

print("Reloaded model input shape:", encoded.shape)
print("Phishing probability:", round(probability, 4))
print("Legitimate probability:", round(1 - probability, 4))

prediction = "phishing" if probability >= 0.5 else "legitimate"
print("Predicted class:", prediction)

print("\nSaved model reload test completed.")


Reloaded model input shape: (1, 300)
Phishing probability: 0.0046
Legitimate probability: 0.9954
Predicted class: legitimate

Saved model reload test completed.


In [21]:

def predict_url(url):
    """Classify a URL using the saved CNN model."""

    if not isinstance(url, str) or not url.strip():
        raise ValueError("Please provide a non-empty URL.")

    url = url.strip()

    # Match the model's training preprocessing
    encoded = reloaded_vectorizer(
        tf.constant([url])
    )

    probability = float(
        reloaded_model(encoded, training=False).numpy()[0][0]
    )

    predicted_class = (
        "phishing" if probability >= 0.5 else "legitimate"
    )

    return {
        "url": url,
        "prediction": predicted_class,
        "phishing_probability": round(probability, 4),
        "legitimate_probability": round(1 - probability, 4),
        "threshold": 0.5
    }


# Test with a sample URL
import json

result = predict_url("https://www.wikipedia.org")
print(json.dumps(result, indent=2))

# Check validation of empty input
try:
    predict_url("   ")
except ValueError as error:
    print("\nInput validation:", error)

print("\nPrediction function test completed.")


{
  "url": "https://www.wikipedia.org",
  "prediction": "legitimate",
  "phishing_probability": 0.0046,
  "legitimate_probability": 0.9954,
  "threshold": 0.5
}

Input validation: Please provide a non-empty URL.

Prediction function test completed.


In [22]:

import os
import numpy as np
import tensorflow as tf

PROJECT_DIR = "/content/phishing_project"

# Save the complete trained CNN model
full_model_path = os.path.join(
    PROJECT_DIR,
    "phishing_url_cnn.keras"
)

model.save(full_model_path)

# Reload the complete model to verify it
loaded_model = tf.keras.models.load_model(
    full_model_path,
    compile=False
)

# Compare predictions from both models
sample = reloaded_vectorizer(
    tf.constant(["https://www.wikipedia.org"])
)

original_probability = float(
    model(sample, training=False).numpy()[0][0]
)

loaded_probability = float(
    loaded_model(sample, training=False).numpy()[0][0]
)

print("Complete model saved successfully.")
print(
    "Model file size (MB):",
    round(os.path.getsize(full_model_path) / (1024 * 1024), 2)
)
print("Original probability:", round(original_probability, 6))
print("Reloaded probability:", round(loaded_probability, 6))

assert np.isclose(
    original_probability,
    loaded_probability,
    atol=1e-6
), "Prediction mismatch after model reload!"

print("\nModel reload verification passed!")


Complete model saved successfully.
Model file size (MB): 0.23
Original probability: 0.004593
Reloaded probability: 0.004593

Model reload verification passed!


In [23]:

import os

PROJECT_DIR = "/content/phishing_project"

folders = [
    "src",
    "artifacts",
    "reports",
    "presentation",
    "tests"
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_DIR, folder),
        exist_ok=True
    )

# Move/copy saved artifacts into an organized folder
import shutil

artifacts = [
    "phishing_url_cnn.keras",
    "best_model.weights.h5",
    "model_config.json",
    "preprocessing_config.json",
    "vocabulary.json",
    "training_history.csv"
]

for filename in artifacts:
    source = os.path.join(PROJECT_DIR, filename)
    destination = os.path.join(
        PROJECT_DIR, "artifacts", filename
    )

    if os.path.exists(source):
        shutil.copy2(source, destination)

print("Project folders created.")
print("\nProject structure:")

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for filename in sorted(files):
        print(f"{indent}  {filename}")


Project folders created.

Project structure:
phishing_project/
  best_model.weights.h5
  model_config.json
  phishing_url_cnn.keras
  preprocessing_config.json
  training_history.csv
  vocabulary.json
  tests/
  artifacts/
    best_model.weights.h5
    model_config.json
    phishing_url_cnn.keras
    preprocessing_config.json
    training_history.csv
    vocabulary.json
  reports/
  presentation/
  src/


In [24]:

import os

PROJECT_DIR = "/content/phishing_project"
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "artifacts")

required_files = [
    "phishing_url_cnn.keras",
    "best_model.weights.h5",
    "model_config.json",
    "preprocessing_config.json",
    "vocabulary.json",
    "training_history.csv"
]

print("Artifact verification:\n")

all_present = True

for filename in required_files:
    path = os.path.join(ARTIFACT_DIR, filename)

    if os.path.isfile(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"OK  {filename} ({size_kb:.2f} KB)")
    else:
        print(f"MISSING  {filename}")
        all_present = False

print("\nAll required artifacts present:", all_present)

assert all_present, "One or more required artifacts are missing."

print("Artifact verification passed.")


Artifact verification:

OK  phishing_url_cnn.keras (240.30 KB)
OK  best_model.weights.h5 (236.73 KB)
OK  model_config.json (5.02 KB)
OK  preprocessing_config.json (0.18 KB)
OK  vocabulary.json (0.67 KB)
OK  training_history.csv (1.00 KB)

All required artifacts present: True
Artifact verification passed.
